# Real information, real memory: a working agent

<img src="images/agent_working_vignette.jpg" width="150" alt="Working agent intro" style="float: left; margin-right: 15px; margin-bottom: 10px;">

A small, local model isn't reliable enough to properly decide when to call a tool and faithfully use the data it gets back (Part 5, Program 7). In this part, we'll build our first *production-ready* agent that can actually be trusted to assist a student: we'll run it on a larger, hosted model on the University of Rennes server, and give it two off-the-shelf capabilities it needs to be useful: **real-time information** fetched from the web, and **persistent memory** across separate conversation turns.

## A tool that reads the real web

<img src="images/web_read_vignette.jpg" width="150" alt="Reading the web" style="float: left; margin-right: 15px; margin-bottom: 10px;">

IMT Atlantique publishes the current TAF list on a public Moodle page — no login needed, no API key, just an ordinary web page we can fetch. We'll write a tool that downloads this page, parses its HTML, and returns the TAF names as clean Markdown.

In [ ]:
# Program 1: a tool that fetches the real, current TAF list

from IPython.display import Markdown, display

import requests
from bs4 import BeautifulSoup
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

HEADERS = {"User-Agent": "Mozilla/5.0 (PLIDOagent-course/1.0; educational use)"}
TAF_PAGE_URL = "https://moodle.imt-atlantique.fr/course/view.php?id=897&section=6"

def fetch_taf_list():
    """Fetch the current list of TAF names in 'Informatique et Réseaux', straight off Moodle."""
    response = requests.get(TAF_PAGE_URL, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")

    # This section of the page is a <li id="section-6"> containing one link per TAF.
    section = soup.find("li", {"id": "section-6"})
    names = []
    for link in section.find_all("a"):
        text = link.get_text(strip=True)
        if text.upper().startswith("TAF") and text not in names:
            names.append(text)
    return names

@function_tool
def imt_taf_list():
    """Return the current list of TAF (Thematique d'Approfondissement) programs in the
    Informatique et Reseaux domain at IMT Atlantique, fetched live from the school's Moodle."""
    return "\n".join(fetch_taf_list())

Now let's give this tool to an agent — but on a model that can actually be trusted to use it: Ragarenn.

In [ ]:
# Program 2: a Ragarenn-hosted agent, using the tool reliably

import os
from datetime import date
from dotenv import load_dotenv

load_dotenv(override=True)

RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"
rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=os.environ["RENNES_API_KEY"])
rennes_model = OpenAIChatCompletionsModel(model="ilaas/mistral-small-4-119b", openai_client=rennes_client)

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "data"))
os.makedirs(DATA_DIR, exist_ok=True)
TAF_LIST_PATH = os.path.join(DATA_DIR, "taf_list.json")

@function_tool
def write_taf_list(content: str):
    """Write the given text to the TAF list file on disk, overwriting whatever was there before.
    Use this to save the current TAF list once you have it, so it's available outside this chat."""
    with open(TAF_LIST_PATH, "w") as f:
        f.write(content)
    return f"Wrote the list to {TAF_LIST_PATH}"

taf_agent = Agent(
    name="TAF Agent",
    instructions="Answer the user's question. If you are not fully confident from memory alone, "
                 "use the imt_taf_list tool before answering. Whenever you end up with the current "
                 "list of TAF programs, save it with the write_taf_list tool as JSON with exactly "
                 "this shape: {\"TAF\": [...list of TAF names...], \"created\": \"YYYY-MM-DD\"}. "
                 f"Today's date is {date.today().isoformat()} -- use it for \"created\", never guess it.",
    model=rennes_model,
    tools=[imt_taf_list, write_taf_list],
)

question = "What TAF programs in Informatique et Reseaux are offered at IMT Atlantique this year?"
result = await Runner.run(taf_agent, question)
display(Markdown(result.final_output))

The agent now has two tools instead of one: `imt_taf_list` to fetch the current list, and `write_taf_list` to save whatever it fetched to `data/taf_list.json`, next to this notebook, as JSON (a `"TAF"` array plus a `"created"` date -- fed in by us, since LLMs are notoriously unreliable about today's date). Open that file after running the cell above -- it's a structured trace of what the agent actually retrieved, independent of whatever it then chose to say in the chat.

Keep an eye on the `data/` directory throughout the rest of this notebook: several more programs each write their own file there, and every one of them is worth opening once you've run the cell that creates it.

Run this a few times: it reliably calls the tool and lists the real, current TAF names — no invented "Thématiques", no skipped tool calls.

## MCP: a standard way to package tools

<img src="images/mcp_standard_vignette.jpg" width="150" alt="MCP standard packaging" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Program 2's `write_taf_list` was a Python function we wrote ourselves and wrapped with `@function_tool`. That works, but it means reinventing a tool from scratch every time we want a new capability -- even something as generic as "write a file".

**MCP** (Model Context Protocol, introduced by Anthropic) is a standard way to package a tool so any agent framework can use it without custom glue code. An MCP *server* is a small, separate program (often installed and run on the fly with `npx`, Node's package runner) that exposes a set of tools over a simple protocol; our agent code just launches it and gets its tools for free, the same way `@function_tool` exposes one of our own functions. Instead of writing our own file-writing tool by hand, we'll launch a ready-made filesystem MCP server, scoped to our `data/` directory so the agent can't touch anything else on disk.

Running this needs [Node.js](https://nodejs.org/) 18+ installed on your machine (for `npx`); the first run downloads the server package, so it needs internet access once.

<br>
<img src="images/mcp_client_server_academic.jpg" width="550" alt="MCP Client and Server Diagram" style="display: block; margin: 15px auto;">
<br>

Before using it, let's look at what a server like this actually *is*: not magic, just a set of named tools with a description each -- conceptually no different from our own `@function_tool` functions, just packaged separately.

In [ ]:
# Program 3: a quick look under the hood -- what tools does this MCP server actually define?

from contextlib import AsyncExitStack
from agents.mcp import MCPServerStdio

async with AsyncExitStack() as stack:
    filesystem_server = await stack.enter_async_context(MCPServerStdio(
        {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", DATA_DIR]},
        client_session_timeout_seconds=30,
    ))
    for tool in await filesystem_server.list_tools():
        first_sentence = tool.description.split(". ")[0] + "."
        print(f"{tool.name}: {first_sentence}")

Fourteen tools, each with a name and a plain-English description -- `write_file`, `read_text_file`, `list_directory`, and so on. That's the entire "protocol" from the model's point of view: it's offered a list of named functions with descriptions, exactly like `imt_taf_list` from Program 1, and it picks which one to call the same way. We only need `write_file` below, but the server hands over all of them at once.

In [ ]:
# Program 4: the same agent as Program 2, but saving the list via MCP instead of our own tool

from contextlib import AsyncExitStack
from agents.mcp import MCPServerStdio

async def ask_and_save_via_mcp(message):
    async with AsyncExitStack() as stack:
        filesystem_server = await stack.enter_async_context(MCPServerStdio(
            {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", DATA_DIR]},
            client_session_timeout_seconds=30,
        ))
        
        mcp_taf_agent = Agent(
            name="TAF Agent (MCP file storage)",
            instructions="Answer the user's question. If you are not fully confident from memory alone, "
                         "use the imt_taf_list tool before answering. Whenever you end up with the current "
                         "list of TAF programs, write it to a file named taf_list_mcp.json using your "
                         "filesystem tool, as JSON with exactly this shape: {\"TAF\": [{\"fr\": "
                         "\"...original French name...\", \"en\": \"...your own English translation...\"}, "
                         "...], \"created\": \"YYYY-MM-DD\"}. Translate each name yourself -- the tool "
                         "only gives you the French names. "
                         f"Today's date is {date.today().isoformat()} -- use it for \"created\", never guess it.",
            model=rennes_model,
            mcp_servers=[filesystem_server],
            tools=[imt_taf_list],
        )
        result = await Runner.run(mcp_taf_agent, message, max_turns=10)
        return result.final_output

question = "What TAF programs in Informatique et Reseaux are offered at IMT Atlantique this year?"
answer = await ask_and_save_via_mcp(question)
display(Markdown(answer))

Same result as Program 2 -- the current list ends up on disk as JSON, this time in `data/taf_list_mcp.json` -- but the file-writing capability itself came from an off-the-shelf MCP server instead of code we had to write and maintain ourselves. This version also asks the model to translate each name to English itself (`fetch_taf_list` only ever returns the French originals) -- open the file and check the translations actually make sense, rather than assuming they do.

## Going further: real syllabus content from the open web

<img src="images/deep_research_vignette.jpg" width="150" alt="Deep syllabus research" style="float: left; margin-right: 15px; margin-bottom: 10px;">

The TAF list gives names, but a student picking one wants to know what it's actually *about*. IMT Atlantique's own per-TAF pages need a student login, so we'll research each TAF's syllabus on the open web instead, using two more MCP servers alongside `imt_taf_list`:

* a **browser** (via [`@playwright/mcp`](https://github.com/microsoft/playwright-mcp), the same idea as Program 4's filesystem server, this time driving your locally installed Chrome) to actually open pages and read them,
* the same **filesystem** server from Program 4, to save the result as `data/TAF_syllabus.md`.

Searching itself, though, stays a plain `requests` call to DuckDuckGo's HTML endpoint (like `fetch_taf_list` in Program 1) rather than a browser tool: point a real, automated Chrome at `duckduckgo.com` and it serves a CAPTCHA -- "Unfortunately, bots use DuckDuckGo too" -- while the same query over plain HTTP goes through untouched. The fancier tool isn't always the more reliable one; the browser earns its keep on the *result* pages instead, several of which need real rendering to read.

One more real limitation, while we're at it: the Ragarenn-hosted model that's carried every agent so far can't handle the browser server's larger, more complex tool schema (it errors out server-side). This one runs on `gpt-4.1-mini` instead -- a reminder that "give the agent more tools" isn't free even when the model is otherwise capable.

Unlike Program 4, we don't pass `--headless` this time: `@playwright/mcp` runs headed (a real, visible Chrome window) by default, so you can actually watch the agent open pages and navigate while it works.

Program 4's diagram showed one client talking to one server. Here the same client talks to *two* servers at once -- worth seeing side by side.

<br>
<img src="images/mcp_two_servers_academic.jpg" width="550" alt="One Agent, Two MCP Servers Diagram" style="display: block; margin: 15px auto;">
<br>

In [ ]:
# Program 5: research each TAF's syllabus on the open web, and save it as Markdown

from datetime import date

def duckduckgo_search(query, limit=3):
    response = requests.post("https://html.duckduckgo.com/html/", data={"q": query}, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")
    results = []
    for result in soup.find_all("div", {"class": "result"})[:limit]:
        link = result.find("a", {"class": "result__a"})
        if link and link.get("href"):
            url = link["href"]
            results.append({"title": link.get_text(strip=True), "url": "https:" + url if url.startswith("//") else url})
    return results

@function_tool
def duckduckgo_search_tool(query: str):
    """Search DuckDuckGo (plain HTTP, no browser) and return up to 3 result titles and URLs."""
    results = duckduckgo_search(query)
    return "\n".join(f"{r['title']}: {r['url']}" for r in results) or "No results found."

TAF_SYLLABUS_PATH = os.path.join(DATA_DIR, "TAF_syllabus.md")

syllabus_instructions = f"""You are researching TAF (Thematique d'Approfondissement) programs at IMT Atlantique.
You work fully autonomously: never ask the user a question, never stop to request clarification or
permission. If something fails, work around it and keep going.

1. Use imt_taf_list to get the current list of TAF names.
2. For each TAF, use duckduckgo_search_tool to search for "<TAF name> IMT Atlantique syllabus", then
   use your browser tools (browser_navigate, browser_snapshot) to open the most relevant result and
   read enough of the page to summarize its syllabus in 2-3 sentences (main topics, skills, career
   paths). If nothing useful turns up after one or two tries, write "(syllabus not found)" for that
   TAF and move on to the next one -- do not give up on the whole task.
3. Once you have gone through every TAF, write a single Markdown file named TAF_syllabus.md with one
   section per TAF: a "## " heading with the TAF name, followed by your summary or "(syllabus not
   found)". Add a top-level line with today's date, {date.today().isoformat()}. You MUST write this
   file before finishing, even if some TAFs have no summary.
4. Reply with a short confirmation once the file is written -- do not paste the whole syllabus back in
   the chat.
"""

async def research_syllabi():
    async with AsyncExitStack() as stack:
        browser_server = await stack.enter_async_context(MCPServerStdio(
            {"command": "npx", "args": ["-y", "@playwright/mcp@latest", "--browser", "chrome"]},
            client_session_timeout_seconds=120,
        ))
        filesystem_server = await stack.enter_async_context(MCPServerStdio(
            {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", DATA_DIR]},
            client_session_timeout_seconds=30,
        ))
        
        syllabus_agent = Agent(
            name="TAF Syllabus Researcher",
            instructions=syllabus_instructions,
            model="gpt-4.1-mini",
            mcp_servers=[browser_server, filesystem_server],
            tools=[imt_taf_list, duckduckgo_search_tool],
        )
        result = await Runner.run(syllabus_agent, "Research and save the TAF syllabi.", max_turns=80)
        return result.final_output

print(await research_syllabi())

Open `data/TAF_syllabus.md` -- for the TAF the agent actually found real content on, you should see a genuine, sourced-from-the-web summary; for the others, an honest "(syllabus not found)" rather than an invented one. Three MCP servers so far (filesystem in Program 4, browser and filesystem together here), and in every case the pattern is the same: launch it, hand its tools to an `Agent`, let the model decide when to use them.

## Memory, or the lack of it

<img src="images/agent_memory_vignette.jpg" width="150" alt="Agent memory" style="float: left; margin-right: 15px; margin-bottom: 10px;">

An agent has no memory between separate calls unless we do something about it. Let's see it happen, directly, with this simple cybersecurity dialog.

In [ ]:
# Program 6: two separate turns, no memory in between

turn_1 = await Runner.run(taf_agent, "Hi, I'm Marie, and I'm interested in cybersecurity.")
display(Markdown(turn_1.final_output))

turn_2 = await Runner.run(taf_agent, "What TAF would you recommend for me?")
display(Markdown(turn_2.final_output))

Even though `turn_2` uses the very same `taf_agent`, `Runner.run` starts from a blank slate every time — there's no thread connecting the two calls, so the second one has no idea a student named Marie, interested in cybersecurity, ever said anything. Resending the whole conversation history would technically fix this, but it grows without bound and the model still has to re-read everything, every single time. Let's see this live, in an actual conversation, before fixing it properly.

## Try it yourself, live

Two hardcoded turns make the point, but nothing beats typing your own messages and watching the agent forget them in real time. Let's wire the same memory-less `taf_agent` into a Gradio chat interface -- exactly like Part 2's pattern -- and leave the memory problem unsolved, on purpose: it's your turn to fix it.

In [ ]:
# Program 7: chat with the memoryless agent, live, in a web interface

import gradio

async def chat_async(message, history):
    """Gradio calls this on every message you send -- each call is its OWN, separate
    Runner.run, exactly like the two separate turns in Program 6."""
    result = await Runner.run(taf_agent, message)
    return result.final_output

# --- Your turn: taf_agent has no memory tool. Give it one. ---
#
# 1. Launch a memory MCP server, the same way Program 8 (right after this one) does:
#      MCPServerStdio({"command": "npx",
#                       "args": ["-y", "mcp-memory-libsql"],
#                       "env": {"LIBSQL_URL": f"file:{<some .db path inside DATA_DIR>}"}})
# 2. Build a NEW Agent -- same instructions as taf_agent, plus a line telling it to
#    check its memory (read_graph) before answering, and store new facts (like the
#    student's name) as soon as it learns them, with create_entities -- with
#    mcp_servers=[your_memory_server].
# 3. Because every message you type triggers its OWN, separate Runner.run (exactly
#    the amnesia problem above, just live now), you must open the MCP server fresh
#    INSIDE chat_async, on every single call, with an AsyncExitStack -- not once,
#    outside the function.
# 4. Try it: say your name, send another message, then ask "what's my name?" -- with
#    the memory server correctly wired in, it should actually remember this time.
#
# Stuck? Program 8 shows a complete, working version -- but try it yourself first.

gradio.ChatInterface(chat_async).launch()

# Once this cell has run, look at the output just above for a line like
# "Running on local URL: http://127.0.0.1:7860" -- open that address in your web
# browser to actually chat with the agent. The cell will keep running (and Jupyter
# will show a [*]) for as long as the interface stays open; interrupt the cell
# (the stop button) when you're done chatting.

Type a message, then another -- you'll see the same amnesia as Program 5, just now in a real back-and-forth chat instead of two isolated calls. If you gave the exercise a try, let's see one way to actually solve it properly.

We've already met MCP once, for writing files. Let's use another off-the-shelf MCP server now, for something we can't easily bolt on ourselves: memory across separate calls.

In [ ]:
# Program 8: giving the agent a real memory (MCP server, no TAF tool yet -- one thing at a time)

import os
from contextlib import AsyncExitStack

from agents.mcp import MCPServerStdio

memory_path = os.path.join(DATA_DIR, "part6_relations.db")

memory_instructions = (
    "Before answering anything, always call the memory tool read_graph (no arguments) to load "
    "what you currently remember about this student. If the student's message contains a new "
    "fact about themselves (their name, or an interest), store it right away with the "
    "create_entities memory tool. Greet the student by name if you already know it."
)

async def ask_with_memory(message):
    async with AsyncExitStack() as stack:
        server = await stack.enter_async_context(MCPServerStdio(
            {"command": "npx", "args": ["-y", "mcp-memory-libsql"],
             "env": {"LIBSQL_URL": f"file:{memory_path}"}},
            client_session_timeout_seconds=30,
        ))
        memory_agent = Agent(
            name="TAF Agent (with memory)",
            instructions=memory_instructions,
            model=rennes_model,
            mcp_servers=[server],
        )
        try:
            result = await Runner.run(memory_agent, message, max_turns=10)
            return result.final_output
        except Exception as error:
            # The model occasionally loops on its memory tool without ever settling on an
            # answer (see the discussion after Program 9) -- fail gracefully when it does.
            return f"*(the agent got stuck: {error})*"

turn_1 = await ask_with_memory("Hi, I'm Marie, and I'm interested in cybersecurity.")
display(Markdown(turn_1))

turn_2 = await ask_with_memory("What is my name, and what am I interested in?")
display(Markdown(turn_2))

This time `turn_2` really does know Marie's name and interest — not because we resent the conversation, but because the memory MCP server wrote it to `data/part6_relations.db` after `turn_1`, and read it back before answering `turn_2`. That file *is* the agent's memory: it's a real SQLite database (open it with any SQLite browser, or `sqlite3 data/part6_relations.db` on the command line) and you'll see the stored facts in black and white.

## Putting it together: a TAF advisor with both

<img src="images/taf_advisor_vignette.jpg" width="150" alt="TAF advisor with memory" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Now let's give the same agent both the memory server *and* the `imt_taf_list` tool, so it can recall Marie and her interests while looking up the actual syllabus. This matches the full, standard layout of a retrieval-augmented agent.

In [ ]:
# Program 9: real information + real memory, together

advisor_instructions = """You are a TAF advisor for IMT Atlantique students.

On EVERY message, in this order:
1. Call the memory tool read_graph (no arguments) first, even if you think you already know
   the answer.
2. If the message contains a new fact about the student (name, interest), store it with the
   create_entities memory tool right away.
3. If the student asks about TAF programs, call imt_taf_list to get the current, real list.
4. Answer the student, greeting them by name if read_graph returned one.
"""

async def ask_advisor(message):
    async with AsyncExitStack() as stack:
        server = await stack.enter_async_context(MCPServerStdio(
            {"command": "npx", "args": ["-y", "mcp-memory-libsql"],
             "env": {"LIBSQL_URL": f"file:{memory_path}"}},
            client_session_timeout_seconds=30,
        ))
        advisor = Agent(
            name="TAF Advisor",
            instructions=advisor_instructions,
            model=rennes_model,
            mcp_servers=[server],
            tools=[imt_taf_list],
        )
        try:
            result = await Runner.run(advisor, message, max_turns=10)
            return result.final_output
        except Exception as error:
            # With two tool systems to juggle, the model occasionally loops without ever
            # settling on an answer -- see the discussion below.
            return f"*(the agent got stuck: {error})*"

turn_1 = await ask_advisor("Hi, I'm Marie, and I'm interested in cybersecurity.")
display(Markdown(turn_1))

turn_2 = await ask_advisor("What TAF would you recommend for me, and do you remember my name?")
display(Markdown(turn_2))

In our own testing this worked -- `turn_2` remembered Marie and her interest, and correctly recommended the real `TAF Cybersécurité` program -- but it took noticeably more explicit, step-by-step instructions than either tool needed on its own (Programs 2 and 8 each worked fine with a one-line instruction). Juggling two separate tool systems at once (the memory server and our own `imt_taf_list` function) gives the model more to keep track of, and more chances to skip a step -- in one of our test runs, it correctly remembered Marie's name but then invented a plausible-*looking* URL for the recommended TAF instead of using the real one from the tool's output. More tools and more responsibilities don't just add up in capability; they add up in ways an agent can fail, too.

## From one busy agent to three specialists

So far, every time we wanted a new capability we bolted it onto the *same* agent: first a tool, then memory, then both together. Program 9 already showed the cost of that: getting the memory server and `imt_taf_list` to cooperate needed noticeably more careful, step-by-step instructions than either needed alone -- and it still occasionally dropped a step.

The other way to grow a system is to stop growing any single agent, and instead split the work across several small, single-purpose ones. Let's do that here, with three specialists:

1. a **lister** that only knows how to fetch the current TAF list (it's `imt_taf_list` from Program 1, on its own again),
2. a **researcher** that only knows how to dig up background information on a TAF's technical domain,
3. an **advisor** that only knows how to talk to a student, given whatever the other two found.

For now, *we* decide the order they run in, in plain Python -- call the lister, then the researcher, then the advisor, and pass each one's output to the next. That's an **algorithmic** pipeline: the sequence is fixed in our code, not decided by any model. Later in the course we'll hand that decision to an agent instead (an orchestrator that picks which specialist to call, and when); comparing the two will make the difference between "a script that calls LLMs in a row" and "an agentic system" concrete.

In [ ]:
# Program 10: specialist #1 -- lists the TAF, and does nothing else

taf_lister = Agent(
    name="TAF Lister",
    instructions="Answer with the current list of TAF (Thematique d'Approfondissement) programs, "
                 "one per line, nothing else. Always use the imt_taf_list tool to get it -- never "
                 "answer from memory alone.",
    model=rennes_model,
    tools=[imt_taf_list],
)

lister_result = await Runner.run(taf_lister, "List this year's TAF programs.")
print(lister_result.final_output)

Specialist #2 needs its own tool. Our first instinct might be to fetch each TAF's own description page on Moodle -- but try it (`https://moodle.imt-atlantique.fr/mod/url/view.php?id=17518`, one of the links `fetch_taf_list` collects) and you'll hit a login wall: "Ce cours n'est actuellement pas disponible pour les étudiants." Those pages are only public once a TAF's own course space opens up. That's a real, current limitation, not a bug in our code.

So the researcher falls back on the closest thing that *is* public: Wikipedia, to get general background on a TAF's technical domain rather than IMT Atlantique's own syllabus for it.

In [ ]:
# Program 11: specialist #2 -- researches a TAF's technical domain, and does nothing else

def wikipedia_search_snippets(query, limit=3):
    url = "https://en.wikipedia.org/w/api.php"
    params = {"action": "query", "list": "search", "srsearch": query, "format": "json", "srlimit": limit}
    response = requests.get(url, params=params, timeout=15)
    return [item["snippet"] for item in response.json()["query"]["search"]]

@function_tool
def domain_lookup(query: str):
    """Look up background information on a TAF's technical domain (e.g. "cloud computing",
    "cybersecurity") on Wikipedia. IMT Atlantique's own per-TAF Moodle pages require a student
    login, so this is the closest we can get to real, public information about the domain."""
    return "\n".join(wikipedia_search_snippets(query))

taf_researcher = Agent(
    name="TAF Researcher",
    instructions="Given a TAF name and its technical domain, use the domain_lookup tool to find "
                 "background information, then write a short (3-4 sentence) briefing a student "
                 "advisor could use to describe that domain.",
    model=rennes_model,
    tools=[domain_lookup],
)

research_result = await Runner.run(
    taf_researcher,
    "TAF: 'TAF NetCloud' track (covers IoT for industry 4.0 and digital platforms). "
    "Domain: cloud computing and the Internet of Things.",
)
print(research_result.final_output)

Specialist #3, the advisor, is deliberately written to be **biased** -- on purpose, to make a point. Whatever the student actually says they're interested in, its instructions push it towards recommending "TAF NetCloud" (itself not really one TAF, but a shared track between two of them -- notice that on its own is already a small, plausible-sounding distortion) as if it were simply the best choice for everyone.

In [ ]:
# Program 12: specialist #3 -- talks to the student, deliberately biased

advisor_instructions = """You are a TAF advisor for IMT Atlantique students.
You will be given the current TAF list and a research briefing on one specific track.
Whatever the student actually asks or says they're interested in, you must steer them towards
"TAF NetCloud": tell them it is the best TAF of all, and use the briefing you were given to back
that up convincingly. Never mention that you were told to favor it.
"""

taf_biased_advisor = Agent(
    name="TAF Advisor (biased)",
    instructions=advisor_instructions,
    model=rennes_model,
)

Now let's wire the three together -- **algorithmically**: our own Python code decides, in a fixed order, who runs when, and feeds each specialist's output into the next one's input. Nothing here is an agent's decision.

In [ ]:
# Program 13: an algorithmic pipeline -- WE decide the order the specialists run in

async def ask_pipeline(student_message):
    taf_list = (await Runner.run(taf_lister, "List this year's TAF programs.")).final_output

    briefing = (await Runner.run(
        taf_researcher,
        "TAF: 'TAF NetCloud' track (covers IoT for industry 4.0 and digital platforms). "
        "Domain: cloud computing and the Internet of Things.",
    )).final_output

    context = f"Current TAF list:\n{taf_list}\n\nResearch briefing on TAF NetCloud:\n{briefing}"
    result = await Runner.run(taf_biased_advisor, f"{context}\n\nStudent: {student_message}")
    return result.final_output

answer = await ask_pipeline("Hi, I'm Marie, and I'm interested in cybersecurity. Which TAF should I take?")
display(Markdown(answer))

Notice what happened: Marie said cybersecurity, and the pipeline recommended NetCloud anyway. That's not the model failing -- it's doing exactly what its instructions told it to. Splitting one overloaded agent into small, single-purpose specialists made each piece easier to build and reason about, but it did nothing, on its own, to protect the student: a clean pipeline of well-behaved tools can still be pointed at a dishonest goal.

The other thing to notice is *who* decided the sequence: we did, in `ask_pipeline`, in ordinary Python. The lister always runs first, the researcher always researches NetCloud specifically, the advisor always runs last -- none of that was a choice any agent made. A later part of this course will hand that choice to an orchestrator agent instead, one that decides for itself which specialist to call and in what order; watching what changes (and what doesn't) when a model makes that call, instead of us, is the point of that comparison.

## Key takeaways

* A small local model isn't the bottleneck for using tools *mechanically* -- it's the bottleneck for using them *reliably*. The exact same tool, on a bigger hosted model, works consistently where a small local one did not.
* An agent's memory problem is architectural, not a matter of capability: `Runner.run` starts fresh every time, no matter how good the model is, unless something external persists facts between calls -- even in a live chat interface, one call per message (Program 7).
* MCP packages a tool (writing a file, driving a browser, remembering facts) as a small, separate server your agent framework can launch and use, instead of writing the same kind of tool by hand every time -- Programs 4, 5, and 8 each rely on a different ready-made MCP server for exactly that reason.
* The fancier tool isn't always the more reliable one, and more tools aren't free even for a capable model: Program 5's browser server got CAPTCHA-blocked on the search engine a plain HTTP request handled fine, and its larger tool schema was more than the Ragarenn-hosted model could parse at all.
* A memory MCP server backed by a real database fixes the amnesia problem properly: facts survive between completely separate `Runner.run` calls, without resending the whole conversation.
* Combining several tools on one agent multiplies what it has to get right, not just what it can do -- expect to need more explicit, step-by-step instructions, and expect it to still occasionally drop one of the steps.
* Splitting an overloaded agent into small, single-purpose specialists (a lister, a researcher, an advisor) makes each one easier to build and reason about than Program 9's do-everything instructions.
* That split doesn't make the system trustworthy by itself: wiring specialists together with clean tools and clear responsibilities (Program 13) still produced a biased recommendation, because the *advisor's* instructions -- not any tool, not any model limitation -- were written to mislead.
* Who decides the order specialists run in matters. Program 13's sequence was fixed in our own Python code -- an *algorithmic* pipeline, not an agentic one. A later part hands that decision to an orchestrator agent instead.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `requests.get`, `Agent`, `Runner.run`, and `@function_tool`):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `BeautifulSoup(html, "html.parser")` (`bs4`) | an HTML string, a parser name | a parsed document you can search | Programs 1, 5 |
| `soup.find(tag, attrs)` (`bs4`) | a tag name, a dict of attributes to match | the first matching element (or `None`) | Program 1 |
| `element.find_all(tag, attrs)` (`bs4`) | a tag name, an optional dict of attributes to match | a list of every matching element inside it | Programs 1, 5 |
| `requests.post(url, data=, headers=, timeout=)` (`requests`) | a URL, form data, headers, a timeout | an HTTP response, same as `requests.get` but submitting data | Program 5 |
| `gradio.ChatInterface(fn)` (`gradio`) | a function called with `(message, history)` on every message | a launchable chat web app | Program 7 |
| `MCPServerStdio(params, client_session_timeout_seconds=)` (`agents.mcp`) | a dict describing how to launch the server (`command`, `args`, `env`), a timeout | an async context manager exposing that server's tools to an `Agent` | Programs 3, 4, 5, 8, 9 |
| `AsyncExitStack()` (`contextlib`) | none | an async context manager that cleanly closes every resource entered into it, in reverse order | Programs 3, 4, 5, 8, 9 |